# Precision Trading System — Full-Scale Training Notebook

**Pipeline:** smoke test → gate on win-rate → full training → evaluation → download.

**Stack:** CatBoost + LightGBM stacking ensemble + isotonic calibration + XGBoost meta-labeler.

Reuses the corrected `TripleBarrierLabeler`, `MarkovRegimeForecaster`, and full feature pipeline from the local repo.

## Sections
1. Environment setup (Colab-ready)
2. Data loading (3 sources: upload / yfinance / Dukascopy)
3. Smoke test on 5–10k bars (gate)
4. Full training (only if smoke passes)
5. Held-out evaluation
6. Save + download model artifact

## 1. Environment setup

Two paths — pick one. Path A clones your repo (preferred); Path B installs only the dependencies and you paste code inline.

In [ ]:
# ── Path A: clone repo (recommended) ──────────────────────────────
# Replace REPO_URL with your repo URL or Drive path.
import os, sys, subprocess

REPO_URL = "https://github.com/YOUR_USERNAME/vibetrading.git"  # ← edit me
REPO_DIR = "/content/vibetrading"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

FRONTEND = os.path.join(REPO_DIR, "trading_terminal", "frontend")
sys.path.insert(0, FRONTEND)
os.chdir(FRONTEND)
print("Frontend on path:", FRONTEND)

In [ ]:
# ── Install dependencies (CPU is fine — no GPU needed for CatBoost/XGBoost/LightGBM) ──
%pip install -q xgboost lightgbm catboost scikit-learn pandas numpy yfinance \
    pykalman hmmlearn joblib

In [ ]:
# Sanity check imports
import warnings; warnings.filterwarnings("ignore")
import os; os.environ["ENABLE_TRADING_MEMORY"] = "false"
import numpy as np, pandas as pd
import xgboost as xgb, lightgbm as lgb
try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("CatBoost unavailable — falling back to XGBoost-only ensemble")

from services.precision_trading_system import (
    PrecisionTradingSystem, Asset,
    triple_barrier_labels_vectorized,
    MarkovRegimeForecaster,
)
from services.training_pipeline import (
    TrainingConfig, TrainingPipeline, DataCleaner,
)
from sklearn.metrics import f1_score, matthews_corrcoef, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV

print("Imports OK · CatBoost:", HAS_CATBOOST)

## 2. Data loading

Pick ONE source. Cell 2A: upload your own CSV. Cell 2B: yfinance (limited to 30 days at 1m). Cell 2C: read the prebuilt `data/historical/<PAIR>_1m.csv`. Cell 2D: Dukascopy (FX/gold/CFDs, free historical 1m, requires `dukascopy_python`).

In [ ]:
# ── 2A: Upload your own OHLCV CSV (timestamp, open, high, low, close, volume) ──
# Skip this cell if using another source.
from google.colab import files
uploaded = files.upload()  # opens browser picker
csv_name = next(iter(uploaded.keys()))
df = pd.read_csv(csv_name)
if "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df = df.set_index("timestamp")
df = df[["open", "high", "low", "close", "volume"]].sort_index().dropna()
print(f"Loaded {len(df):,} bars from {csv_name}")
print(df.head())

In [ ]:
# ── 2B: yfinance — last 30 days of 1m, or longer at 1h/1d ──
import yfinance as yf

YF_TICKER = "GC=F"     # gold futures (use BTC-USD / EURUSD=X / GBPUSD=X / etc.)
INTERVAL = "1m"        # 1m max ~30d, 1h max ~730d, 1d max years
PERIOD = "30d"

df = yf.download(YF_TICKER, period=PERIOD, interval=INTERVAL,
                 progress=False, auto_adjust=True, threads=False)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
df.columns = [c.lower() for c in df.columns]
df = df[["open", "high", "low", "close", "volume"]].dropna()
df.index = pd.to_datetime(df.index, utc=True); df.index.name = "timestamp"
print(f"Loaded {len(df):,} bars · {df.index[0]} → {df.index[-1]}")

In [ ]:
# ── 2C: prebuilt CSV from the repo (already cleaned + multi-source merged) ──
PAIR = "XAUUSD"          # XAUUSD / BTCUSD / EURUSD / GBPUSD
TIMEFRAME = "1m"        # 1m | 1h | daily
csv = f"data/historical/{PAIR}_{TIMEFRAME}.csv"
df = pd.read_csv(csv, index_col="timestamp", parse_dates=["timestamp"])
if "source" in df.columns:
    df = df[df["source"] == "yfinance"]  # keep only real OHLC, drop ECB-derived close-only rows
df = df[["open", "high", "low", "close", "volume"]].sort_index().dropna()
print(f"Loaded {len(df):,} bars · {df.index[0]} → {df.index[-1]}")

In [ ]:
# ── 2D (optional): Dukascopy 1m for FX/gold (years of free history) ──
# %pip install -q dukascopy_python
# from dukascopy_python.instruments import INSTRUMENT_FX_MAJORS_EUR_USD
# from dukascopy_python import fetch, INTERVAL_MIN_1, OFFER_SIDE_BID
# from datetime import datetime, timedelta
# end = datetime.now(); start = end - timedelta(days=540)
# df = fetch(INSTRUMENT_FX_MAJORS_EUR_USD, INTERVAL_MIN_1, OFFER_SIDE_BID, start, end)
# df = df.rename(columns=str.lower)[["open","high","low","close","volume"]]

## 3. Smoke test (gate)

Train on the first 70% of a 5–10k-bar slice, evaluate on the last 30%. **If OOS win-rate < 48% we abort** — no point burning hours on full training when the small slice already shows no edge. The threshold is conservative; published 1m FX SOTA is 55–60%.

In [ ]:
# Smoke test config
ASSET = "XAUUSD"          # must match the data
SMOKE_BARS = 8000          # quick slice
SMOKE_TRAIN_PCT = 0.70
SMOKE_WR_THRESHOLD = 0.48  # gate: abort full training below this
SMOKE_F1_THRESHOLD = 0.18

smoke_df = df.iloc[:SMOKE_BARS].copy() if len(df) > SMOKE_BARS else df.copy()
if smoke_df.index.tz is None:
    smoke_df.index = smoke_df.index.tz_localize("UTC")
print(f"Smoke slice: {len(smoke_df):,} bars · {smoke_df.index[0]} → {smoke_df.index[-1]}")

In [ ]:
# Build features + smoke-train the default XGBoost
sys_smoke = PrecisionTradingSystem(asset=Asset(ASSET), model_type="xgboost", use_hmm=True)
split = int(len(smoke_df) * SMOKE_TRAIN_PCT)
sys_smoke.train(smoke_df.iloc[:split])

# Build OOS features + predict
test_feat = sys_smoke._build_features(smoke_df.iloc[split:])
preds = sys_smoke.signal_model.predict(test_feat)
y_true = triple_barrier_labels_vectorized(
    test_feat,
    pt_atr_mult=getattr(sys_smoke.config, "atr_multiplier_tp1", 1.5),
    sl_atr_mult=getattr(sys_smoke.config, "atr_multiplier_stop", 1.0),
    max_bars=10,
)
y_pred = preds.reindex(y_true.index).fillna(0).values
yt = y_true.values
mask = ~(np.isnan(yt) | np.isnan(y_pred.astype(float)))
smoke_f1 = f1_score(yt[mask], y_pred[mask], average="macro", zero_division=0)
smoke_acc = (yt[mask] == y_pred[mask]).mean()
nz = y_pred[mask] != 0
smoke_wr = (np.sign(yt[mask][nz]) == np.sign(y_pred[mask][nz])).mean() if nz.sum() else 0.0
smoke_mcc = matthews_corrcoef(yt[mask], y_pred[mask])

print(f"\n── SMOKE RESULTS ──")
print(f"F1 macro:   {smoke_f1:.3f}  (threshold {SMOKE_F1_THRESHOLD:.2f})")
print(f"Accuracy:   {smoke_acc:.1%}")
print(f"Win rate:   {smoke_wr:.1%}  (threshold {SMOKE_WR_THRESHOLD:.0%})")
print(f"MCC:        {smoke_mcc:.3f}")
print(f"Pred dist:  {pd.Series(y_pred[mask]).value_counts().sort_index().to_dict()}")
print(f"True dist:  {pd.Series(yt[mask]).value_counts().sort_index().to_dict()}")

In [ ]:
# ── GATE: abort if smoke fails ──
PROCEED = (smoke_wr >= SMOKE_WR_THRESHOLD) and (smoke_f1 >= SMOKE_F1_THRESHOLD)

if not PROCEED:
    print("\n❌ SMOKE FAILED — not proceeding to full training.")
    print("Try: more data, different asset, different label horizon, or check")
    print("data quality (gaps, outliers, time-zone misalignment).")
    raise SystemExit("Smoke gate failed")

print("\n✅ SMOKE PASSED — proceeding to full training.")

## 4. Full training — CatBoost + LightGBM stacking ensemble

Stacking architecture (CPU-friendly, no GPU needed):
1. **CatBoost** on tabular microstructure features (handles categoricals natively).
2. **LightGBM** as a parallel base learner.
3. **XGBoost** as the third base learner.
4. **Logistic regression meta-learner** stacks the three base models' probabilities.
5. **Isotonic calibration** wraps the final stacked output.

Trained with temporal-decay sample weights via `TrainingPipeline` for the data-cleaning + 3-way split, then base learners are fitted manually so we can stack them properly.

In [ ]:
# Run the existing TrainingPipeline (data clean + split + features + Markov)
from services.training_pipeline import TrainingConfig, TrainingPipeline

if df.index.tz is None:
    df.index = df.index.tz_localize("UTC")

# Use the full df as the source — pipeline will split internally
def fetch_fn(start, end, interval="1m"):
    s = pd.Timestamp(start, tz="UTC") if not hasattr(start, "tz") or start.tz is None else start
    e = pd.Timestamp(end, tz="UTC") if not hasattr(end, "tz") or end.tz is None else end
    return df.loc[(df.index >= s) & (df.index <= e)]

# Adjust window to actual data extent
total_days = (df.index[-1] - df.index[0]).days
test_days = max(5, total_days // 10)
val_days = max(3, total_days // 14)
train_days = total_days - test_days - val_days - 1
print(f"Window: train={train_days}d val={val_days}d test={test_days}d")

cfg = TrainingConfig(
    asset=ASSET, model_type="xgboost",
    train_days=train_days, validation_days=val_days, test_days=test_days,
    interval="1m" if (df.index[1] - df.index[0]).total_seconds() < 600 else "1h",
    min_bars_per_day=20,
    temporal_decay_halflife=720,  # ~12h half-life on 1m data
)
sys_full = PrecisionTradingSystem(asset=Asset(ASSET), model_type="xgboost", use_hmm=True)
pipeline = TrainingPipeline(sys_full, cfg)
metrics_xgb = pipeline.run(fetch_fn)
print(f"\n── XGBoost baseline (TrainingPipeline) ──")
print(f"  Train F1={metrics_xgb.train_f1:.3f}  Val F1={metrics_xgb.val_f1:.3f}  OOS F1={metrics_xgb.oos_f1:.3f}")
print(f"  OOS WR={metrics_xgb.oos_winrate:.1%}  OOS Acc={metrics_xgb.oos_accuracy:.1%}  OOS MCC={metrics_xgb.oos_mcc:.3f}")
print(f"  Saved: {metrics_xgb.model_path}")

In [ ]:
# Now build the stacking ensemble on top.
# Reuse the engineered features by re-running _build_features on full df.
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV

feats_full = sys_full._build_features(df)
feature_cols = [c for c in feats_full.columns
                if c not in ["open", "high", "low", "close", "volume"]]
X_all = feats_full[feature_cols].fillna(0).values
y_all = triple_barrier_labels_vectorized(
    feats_full,
    pt_atr_mult=getattr(sys_full.config, "atr_multiplier_tp1", 1.5),
    sl_atr_mult=getattr(sys_full.config, "atr_multiplier_stop", 1.0),
    max_bars=10,
).values
n = len(X_all)
test_n = int(n * test_days / total_days)
val_n = int(n * val_days / total_days)
X_train, y_train = X_all[: -(test_n + val_n)], y_all[: -(test_n + val_n)]
X_val,   y_val   = X_all[-(test_n + val_n) : -test_n], y_all[-(test_n + val_n) : -test_n]
X_test,  y_test  = X_all[-test_n :], y_all[-test_n :]
print(f"Train/Val/Test: {len(X_train)} / {len(X_val)} / {len(X_test)}")

# Encode labels {-1, 0, +1} → {0, 1, 2} for sklearn
label_map = {-1: 0, 0: 1, 1: 2}
inv_map = {v: k for k, v in label_map.items()}
y_train_e = np.array([label_map.get(int(y), 1) for y in y_train])
y_val_e = np.array([label_map.get(int(y), 1) for y in y_val])
y_test_e = np.array([label_map.get(int(y), 1) for y in y_test])

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

In [ ]:
# ── Train base learners with temporal decay sample weights ──
halflife = 720
decay_lambda = np.log(2) / halflife
t = np.arange(len(X_train))
weights = np.exp(decay_lambda * (t - len(X_train)))
weights = weights / weights.sum() * len(X_train)

base_preds_val = []
base_preds_test = []
base_models = {}

# 1) XGBoost
print("Training XGBoost…")
xgb_clf = xgb.XGBClassifier(
    n_estimators=2000, max_depth=6, learning_rate=0.01,
    reg_lambda=5.0, min_child_weight=20,
    subsample=0.9, colsample_bytree=0.9,
    tree_method="hist", random_state=42,
    early_stopping_rounds=100, eval_metric="mlogloss",
)
xgb_clf.fit(X_train_s, y_train_e, sample_weight=weights,
            eval_set=[(X_val_s, y_val_e)], verbose=False)
base_preds_val.append(xgb_clf.predict_proba(X_val_s))
base_preds_test.append(xgb_clf.predict_proba(X_test_s))
base_models["xgb"] = xgb_clf

# 2) LightGBM
print("Training LightGBM…")
lgb_clf = lgb.LGBMClassifier(
    n_estimators=2000, max_depth=8, learning_rate=0.01,
    reg_lambda=5.0, min_child_samples=30,
    subsample=0.9, colsample_bytree=0.9,
    random_state=42, verbose=-1, n_jobs=-1,
)
lgb_clf.fit(X_train_s, y_train_e, sample_weight=weights,
            eval_set=[(X_val_s, y_val_e)],
            callbacks=[lgb.early_stopping(100, verbose=False)])
base_preds_val.append(lgb_clf.predict_proba(X_val_s))
base_preds_test.append(lgb_clf.predict_proba(X_test_s))
base_models["lgb"] = lgb_clf

# 3) CatBoost (if available)
if HAS_CATBOOST:
    print("Training CatBoost…")
    cb_clf = CatBoostClassifier(
        iterations=2000, depth=6, learning_rate=0.01,
        l2_leaf_reg=5.0, random_seed=42, verbose=False,
        early_stopping_rounds=100,
    )
    cb_clf.fit(X_train_s, y_train_e, sample_weight=weights,
               eval_set=(X_val_s, y_val_e), use_best_model=True)
    base_preds_val.append(cb_clf.predict_proba(X_val_s))
    base_preds_test.append(cb_clf.predict_proba(X_test_s))
    base_models["cat"] = cb_clf

print(f"\n{len(base_models)} base learners trained.")
for name, m in base_models.items():
    p = m.predict(X_val_s)
    print(f"  {name}: val_acc={(p == y_val_e).mean():.3f}")

In [ ]:
# ── Stacking meta-learner: logistic regression on base-learner probabilities ──
# Concatenate base probs as features for the meta-learner.
M_train_meta = np.hstack(base_preds_val)
M_test_meta = np.hstack(base_preds_test)
print(f"Meta-features: {M_train_meta.shape[1]} dims (n_bases × n_classes)")

meta = LogisticRegression(
    multi_class="multinomial", solver="lbfgs",
    C=1.0, max_iter=200, random_state=42,
)
meta.fit(M_train_meta, y_val_e)

# ── Isotonic calibration on top of meta ──
calib_meta = CalibratedClassifierCV(meta, method="isotonic", cv="prefit")
calib_meta.fit(M_train_meta, y_val_e)

# Final predictions on test set
y_pred_test_e = calib_meta.predict(M_test_meta)
y_pred_test = np.array([inv_map[int(p)] for p in y_pred_test_e])
y_test_orig = np.array([inv_map[int(t)] for t in y_test_e])

stack_f1 = f1_score(y_test_orig, y_pred_test, average="macro", zero_division=0)
stack_acc = (y_test_orig == y_pred_test).mean()
nz = y_pred_test != 0
stack_wr = (np.sign(y_test_orig[nz]) == np.sign(y_pred_test[nz])).mean() if nz.sum() else 0.0
stack_mcc = matthews_corrcoef(y_test_orig, y_pred_test)

print(f"\n── STACKED ENSEMBLE — OOS RESULTS ──")
print(f"F1 macro:   {stack_f1:.3f}")
print(f"Accuracy:   {stack_acc:.1%}")
print(f"Win rate:   {stack_wr:.1%}")
print(f"MCC:        {stack_mcc:.3f}")
print(f"Pred dist:  {pd.Series(y_pred_test).value_counts().sort_index().to_dict()}")
print(f"True dist:  {pd.Series(y_test_orig).value_counts().sort_index().to_dict()}")

## 5. Held-out evaluation: stacking vs plain XGBoost

Compare the stacked ensemble against the single XGBoost baseline so you know whether stacking actually helped on YOUR data. If stacking doesn't beat single XGBoost by ≥1pp on win-rate, ship the simpler model.

In [ ]:
# Plain XGBoost baseline metrics on the same test slice
y_pred_xgb_e = xgb_clf.predict(X_test_s)
y_pred_xgb = np.array([inv_map[int(p)] for p in y_pred_xgb_e])
xgb_f1 = f1_score(y_test_orig, y_pred_xgb, average="macro", zero_division=0)
xgb_acc = (y_test_orig == y_pred_xgb).mean()
nz = y_pred_xgb != 0
xgb_wr = (np.sign(y_test_orig[nz]) == np.sign(y_pred_xgb[nz])).mean() if nz.sum() else 0.0

print("            F1     Acc     WR      MCC")
print(f"XGBoost   {xgb_f1:.3f}  {xgb_acc:.1%}  {xgb_wr:.1%}")
print(f"Stacked   {stack_f1:.3f}  {stack_acc:.1%}  {stack_wr:.1%}  {stack_mcc:.3f}")

wr_lift = stack_wr - xgb_wr
f1_lift = stack_f1 - xgb_f1
print(f"\nStacking lift: WR {wr_lift:+.1%}, F1 {f1_lift:+.3f}")

USE_STACK = wr_lift >= 0.01 and f1_lift >= 0
FINAL_MODEL = "stacked_ensemble" if USE_STACK else "xgboost_only"
print(f"→ Recommend: {FINAL_MODEL}")

## 6. Save + download the trained artifact

Saves a single bundle (`*.pkl`) containing all base models, scaler, meta-learner, calibrator, label map, and feature columns. You can drop this into the local repo as the production model.

In [ ]:
import joblib, json
from datetime import datetime

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = f"/content/precision_models_{ASSET}"
os.makedirs(out_dir, exist_ok=True)
model_path = f"{out_dir}/precision_{ASSET}_{FINAL_MODEL}_{ts}.pkl"

bundle = {
    "asset": ASSET,
    "final_model": FINAL_MODEL,
    "feature_cols": feature_cols,
    "scaler": scaler,
    "label_map": label_map,
    "inv_map": inv_map,
    "base_models": base_models if USE_STACK else None,
    "meta": meta if USE_STACK else None,
    "calibrated_meta": calib_meta if USE_STACK else None,
    "single_xgb": xgb_clf if not USE_STACK else None,
    "trained_at": datetime.now().isoformat(),
    "metrics_xgb_only": {"f1": float(xgb_f1), "acc": float(xgb_acc), "wr": float(xgb_wr)},
    "metrics_stacked": {"f1": float(stack_f1), "acc": float(stack_acc),
                          "wr": float(stack_wr), "mcc": float(stack_mcc)} if USE_STACK else None,
    "label_horizon_bars": 10,
    "data_range": [str(df.index[0]), str(df.index[-1])],
    "n_train": int(len(X_train)), "n_val": int(len(X_val)), "n_test": int(len(X_test)),
}
joblib.dump(bundle, model_path, compress=3)

meta_path = model_path.replace(".pkl", "_meta.json")
with open(meta_path, "w") as f:
    json.dump({k: v for k, v in bundle.items()
                if k not in ("scaler", "base_models", "meta",
                              "calibrated_meta", "single_xgb")}, f, indent=2, default=str)

print(f"Saved bundle: {model_path}")
print(f"Saved meta:   {meta_path}")
size_mb = os.path.getsize(model_path) / 1024 / 1024
print(f"Bundle size: {size_mb:.1f} MB")

In [ ]:
# Download the bundle locally
from google.colab import files
files.download(model_path)
files.download(meta_path)

## 7. Quick in-notebook test on the saved bundle

Load the bundle back and run inference on the last few hundred bars to confirm it deserialises correctly and predicts as expected.

In [ ]:
import joblib
loaded = joblib.load(model_path)

# Predict on the last 200 bars
tail = df.tail(200)
tail_feat = sys_full._build_features(tail)
X_tail = scaler.transform(tail_feat[loaded["feature_cols"]].fillna(0).values)

if loaded["final_model"] == "stacked_ensemble":
    base_probs = []
    for name, m in loaded["base_models"].items():
        base_probs.append(m.predict_proba(X_tail))
    M_tail = np.hstack(base_probs)
    enc_preds = loaded["calibrated_meta"].predict(M_tail)
else:
    enc_preds = loaded["single_xgb"].predict(X_tail)

preds = np.array([loaded["inv_map"][int(p)] for p in enc_preds])
print("Tail predictions (last 200 bars):")
print(pd.Series(preds).value_counts().sort_index().to_dict())
print("Last 10 predictions:", preds[-10:].tolist())

## 8. (Optional) Drop the bundle into the local repo

On your local machine after downloading:

```bash
cp ~/Downloads/precision_XAUUSD_*.pkl trading_terminal/frontend/data/models/XAUUSD/
```

Then write a small loader in `services/precision_trading_system.py` that detects the bundle dict shape (vs the legacy single-model pickle) and wires it as `signal_model`. Or — simpler — keep using the production pipeline locally and treat the Colab-trained bundle as a research artifact for comparison.

## What to do if metrics still aren't where you want

1. **More data.** 30 days of yfinance 1m on FX is too little. Use the prebuilt `data/historical/<PAIR>_1m.csv` (60 days) or Dukascopy (years).
2. **Different label horizon.** Try `max_bars=5` (faster signals) or `max_bars=20` (longer holds). The optimum is asset-dependent.
3. **Different barriers.** Symmetric `pt=sl=1.5×ATR` is balanced; if shorts dominate, increase `sl_mult`.
4. **Cost-aware labels.** Add `cost_pct=0.0002` (gold) / `0.0008` (BTC) to penalise tight wins that costs eat.
5. **Skip the BiLSTM.** Tabular boosted trees consistently beat sequence models on 1m FX/gold (Borrageiro 2022, Tashiro 2023). More data > more model.